# 01 - Extract

**Stage:** Extract (the *E* in ETL).

Responsibilities of this notebook:
- Load the **immutable raw** source data from `data/raw/`.
- Perform *only* lightweight validation (shape, columns, dtypes).
- Do **not** clean or transform here — that belongs in `02_transform`.

> Treat `data/raw/` as read-only. Never overwrite source files.

In [1]:
import os
import sys

path = os.getcwd()
path = os.path.abspath(os.path.join(path, "..", "data"))
PROJECT_ROOT = os.path.abspath(os.path.join(path, ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
path


'/Users/marcel/Documents/github/property-near-the-beach-predictor/data'

## Load raw data

In [2]:
from pathlib import Path
import pandas as pd

from src.dataset_filters import ALLOWED_NOT_BEACH_CATEGORIES, filter_property_relevant_manifest, infer_not_beach_category

RAW_DIR = Path(path) / "raw"
BEACH_DIR = RAW_DIR / "beach"
NOT_BEACH_DIR = RAW_DIR / "not_beach"
INTERIM_DIR = Path(path) / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = Path(path).parent

records = []
for class_name, label, folder in [("beach", 1, BEACH_DIR), ("not_beach", 0, NOT_BEACH_DIR)]:
    for fp in sorted(folder.rglob("*.jpg")):
        records.append({
            "filepath": str(fp.relative_to(PROJECT_ROOT)),
            "filename": fp.name,
            "label": label,
            "class_name": class_name,
        })

df_raw = pd.DataFrame(records)
print(f"Total images found before filtering: {len(df_raw)}")

not_beach_categories = df_raw.loc[df_raw["label"] == 0, "filepath"].map(infer_not_beach_category)
print("Top not_beach categories before filtering:")
print(not_beach_categories.value_counts().head(15).to_string())

df_raw = filter_property_relevant_manifest(df_raw)
print(f"\nTotal images kept after property-only filtering: {len(df_raw)}")
print(f"Allowed not_beach categories: {sorted(ALLOWED_NOT_BEACH_CATEGORIES)}")
df_raw.head()


Total images found: 27624


,filepath,filename,label,class_name
0,data/raw/beach/i0001.jpg,i0001.jpg,1,beach
1,data/raw/beach/i0002.jpg,i0002.jpg,1,beach
2,data/raw/beach/i0003.jpg,i0003.jpg,1,beach
3,data/raw/beach/i0004.jpg,i0004.jpg,1,beach
4,data/raw/beach/i0005.jpg,i0005.jpg,1,beach


## Validation & profiling

In [3]:
assert not df_raw[df_raw["class_name"] == "beach"].empty, "No beach images found in data/raw/beach/"
assert not df_raw[df_raw["class_name"] == "not_beach"].empty, "No not_beach images found after filtering"

missing = [fp for fp in df_raw["filepath"] if not (PROJECT_ROOT / fp).exists()]
assert not missing, f"{len(missing)} file(s) not found on disk: {missing[:5]}"

not_beach_categories = df_raw.loc[df_raw["label"] == 0, "filepath"].map(infer_not_beach_category)
assert set(not_beach_categories.unique()).issubset(ALLOWED_NOT_BEACH_CATEGORIES), (
    "Unexpected non-property categories still present in the negative class"
)

print("Both classes present")
print("All files readable")
print("Negative class limited to property-style listing imagery\n")
print("Class counts:")
print(df_raw["class_name"].value_counts().to_string())
print("\nFiltered not_beach category counts:")
print(not_beach_categories.value_counts().to_string())


✓ Both classes present
✓ All files readable

Class counts:
class_name
not_beach    24907
beach         2717


In [4]:
# Random sample from the manifest
df_raw.sample(min(10, len(df_raw)), random_state=42)

,filepath,filename,label,class_name
20386,data/raw/not_beach/officebuilding_0462.jpg,officebuilding_0462.jpg,0,not_beach
25627,data/raw/not_beach/roof_0395.jpg,roof_0395.jpg,0,not_beach
9061,data/raw/not_beach/cleveland_BdIdx_1313_BdType...,cleveland_BdIdx_1313_BdType_retail.jpg,0,not_beach
13167,data/raw/not_beach/house_0637.jpg,house_0637.jpg,0,not_beach
26636,data/raw/not_beach/roof_1404.jpg,roof_1404.jpg,0,not_beach
21468,data/raw/not_beach/officebuilding_1544.jpg,officebuilding_1544.jpg,0,not_beach
14653,data/raw/not_beach/house_2123.jpg,house_2123.jpg,0,not_beach
21003,data/raw/not_beach/officebuilding_1079.jpg,officebuilding_1079.jpg,0,not_beach
23379,data/raw/not_beach/retail_0347.jpg,retail_0347.jpg,0,not_beach
7417,data/raw/not_beach/church_0559.jpg,church_0559.jpg,0,not_beach


## Save manifest

In [5]:
MANIFEST_FILE = INTERIM_DIR / "image_manifest.parquet"
df_raw.to_parquet(MANIFEST_FILE, index=False)
print(f"Saved manifest → {MANIFEST_FILE}")
print(f"Rows: {len(df_raw)}  |  Columns: {list(df_raw.columns)}")


Saved manifest → /Users/marcel/Documents/github/property-near-the-beach-predictor/data/interim/image_manifest.parquet
Rows: 27624  |  Columns: ['filepath', 'filename', 'label', 'class_name']
